# ImamAli110 — Manual Quality Test 001
آزمایش دستی و مستقل کیفیت؛ بدون GitHub Actions و بدون آپلود یوتیوب. این فایل فقط موتور `viral_short.py` را از شاخهٔ آزمایشی می‌گیرد و خروجی را برای بررسی دستی می‌سازد.

**اجرا:** Runtime → Change runtime type → T4 GPU، سپس سلول‌ها را به ترتیب اجرا کن. اگر Colab تخصیص GPU را رد کرد، نوت‌بوک جدید هم نمی‌تواند سهمیه را دور بزند.

In [ ]:
import torch, os, subprocess, sys
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU در این نشست فعال نیست؛ از Runtime → Change runtime type یک GPU انتخاب کن.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
# دریافت دقیق شاخهٔ آزمایشی؛ نه main. اجرای مجدد کد را روی همین نسخه نگه می‌دارد.
REPO='/content/Youtube1-quality-test-001'
BRANCH='fix/viral-short-qc-recovery'
if not os.path.isdir(REPO+'/.git'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/farzin0369/Youtube1.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH],check=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','fonts-noto-core','fontconfig'],check=True)
subprocess.run(['fc-cache','-f'],check=True)
print('Engine:', REPO+'/engine/viral_short.py')
print('Branch:', BRANCH)

## خروجی آزمایشی
سلول بعدی یک کلیپ فارسی کوتاه می‌سازد. هیچ اتصال یا کلید یوتیوبی لازم نیست و هیچ چیزی منتشر نمی‌شود. اولین اجرا زمان‌بر است چون مدل SDXL دانلود می‌شود.

In [ ]:
OUT='/content/ImamAli110_test_001.mp4'
WORK='/content/ImamAli110_test_001_work'
cmd=[sys.executable, REPO+'/engine/viral_short.py','--out',OUT,'--work',WORK]
print('Starting manual render; no upload will occur.')
subprocess.run(cmd,check=True)
assert os.path.isfile(OUT) and os.path.getsize(OUT)>1_000_000, 'خروجی ویدئو ساخته نشد یا ناقص است.'
print('VIDEO READY:',OUT,'size MB:',round(os.path.getsize(OUT)/1024**2,2))

In [ ]:
# بررسی فنی و دانلود خروجی برای مشاهدهٔ دستی
subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size','-of','default=noprint_wrappers=1',OUT],check=True)
from google.colab import files
files.download(OUT)

## معیار تأیید کیفیت
- نوشتهٔ فارسی خوانا و درست نمایش داده شود.
- تصاویر با روایت هماهنگ باشند و چهره‌ها/بدن‌ها ایراد واضح نداشته باشند.
- صدای فارسی طبیعی باشد و با متن هماهنگی داشته باشد.
- اگر هر مورد رد شد، نتیجه را ثبت می‌کنیم و در نوت‌بوک آزمایش بعدی فقط اصلاح مشخص همان ایراد را می‌سنجیم.

**توجه:** موتور فعلی از تصویرهای ثابت سینمایی با حرکت دوربین و صدای فارسی استفاده می‌کند؛ ویدئوی مولد واقعی از مدل text-to-video نیست.